# Price-prediction neural network

Walkthrough of the Keras model that predicts the sale price of a Madrid flat
from six numeric features. It mirrors `model.py`; see also `recommend_price.py`
for the tuned, benchmarked variant.

## Imports

The model is built with Keras (TensorFlow backend); pandas/NumPy handle the
dataset.

In [ ]:
import keras
import numpy as np
import pandas as pd
from keras.layers import Dense, Input
from keras.models import Sequential

keras.utils.set_random_seed(42)

## Load the dataset

Drop the categorical columns, fill missing values with `1`, then split into the
six predictors `X` (`Precio_m2`, `Habitaciones`, `Aseos`, `Superficie`,
`Parking`, `Colegios`) and the target `y` (`Precio`).

In [ ]:
csv_route = "finalDataset3.csv"

raw = pd.read_csv(csv_route, header=0, encoding="latin1")
data = raw.fillna(value=1).drop(["Tipo", "Distrito"], axis=1).to_numpy()

X = data[:, 2:8].astype("float32")
y = data[:, 1].astype("float32")
X.shape, y.shape

## Build the network

Six inputs, one output. Here we use four hidden `Dense(6, relu)` layers; add or
remove `model.add(Dense(...))` calls to change the depth.

In [ ]:
model = Sequential(
    [
        Input(shape=(6,)),
        Dense(6, activation="relu"),
        Dense(6, activation="relu"),
        Dense(6, activation="relu"),
        Dense(6, activation="relu"),
        Dense(1, activation="relu"),
    ]
)
model.summary()

## Compile

Swap the optimizer (`keras.optimizers.SGD` / `RMSprop` / `Adam`), its
`learning_rate`, or the `loss` (`mean_squared_logarithmic_error`, `huber`,
`log_cosh`, ...) to experiment.

In [ ]:
optimizer = keras.optimizers.SGD(learning_rate=0.1)
model.compile(loss="mean_squared_logarithmic_error", optimizer=optimizer, metrics=["msle"])

## Train

Raise `epochs` for a longer run; `validation_split` holds back part of the data
to watch for overfitting.

In [ ]:
history = model.fit(X, y, validation_split=0.30, epochs=150, batch_size=120)
print("Minimum training MSLE:", float(np.min(history.history["msle"])))